# 动量语义拆分：可逆、不拟合
保留原3/5/10/20/60窗口端点，将重叠收益拆成非重叠历史段。仅检查信息保留与输入相关性，不看未来收益，不替换冻结网络输入。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.workflow.minute_action_value import _source
from src.data.allocation_feature_contract import canonical_return_inputs,disjoint_momentum_inputs,reconstruct_momentum,MOMENTUM_HORIZONS
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/momentum_semantics_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
source,audit=_source(root,plan['source'],['inputs/features','inputs/common','inputs/frozen_parent_models'])
state=source['inputs/frozen_parent_models']['outer_refit'];primary=source['inputs/features'][state['primary_features']];common=source['inputs/common'][state['common_features']].reindex(primary.index)
canonical,aliases=canonical_return_inputs(primary,common);segmented,mapping=disjoint_momentum_inputs(canonical)
# Consume the research representation through a native Qlib Dataset.
grouped=pd.concat({'feature':segmented},axis=1);grouped.index=pd.MultiIndex.from_arrays([grouped.index,['SOXL_SOXX_CASH']*len(grouped)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(grouped),segments={'all':(None,None)})
observed=dataset.prepare('all',col_set='feature').droplevel('instrument');observed.index.name=canonical.index.name
pd.testing.assert_frame_equal(observed,segmented)
restored=reconstruct_momentum(observed);original=canonical[restored.columns]
np.testing.assert_allclose(restored,original,rtol=1e-12,atol=1e-12,equal_nan=True)
other=canonical.columns.difference(original.columns);pd.testing.assert_frame_equal(observed[other],canonical[other])
rows=[]
for window,start,end in [('training_history',None,'2022-12-30'),('reused_2023_2026','2023-01-03',None)]:
 for asset in ['soxl','soxx']:
  oldcols=[f'primary:{asset}_ret{h}' for h in MOMENTUM_HORIZONS];newcols=list(mapping.loc[mapping.source.isin(oldcols),'target'])
  for role,frame in [('overlapping_log_returns',np.log1p(canonical[oldcols])),('disjoint_log_returns',observed[newcols])]:
   part=frame.loc[start:end].dropna();matrix=part.corr().to_numpy();pairs=matrix[np.triu_indices(5,k=1)]
   rows.append({'window':window,'asset':asset,'representation':role,'observations':len(part),'mean_absolute_pair_correlation':float(np.abs(pairs).mean()),'maximum_absolute_pair_correlation':float(np.abs(pairs).max())})
anchor=observed.loc[[plan['anchor']],mapping.target].T.reset_index();anchor.columns=['feature','log_return'];anchor['equivalent_segment_return_pct']=100*np.expm1(anchor.log_return)
summary={'experiment':card.name,'decision':'lossless_research_representation_only_no_alpha_claim','new_fits':0,'new_backtests':0,'parameter_searches':0,'outcome_correlations':0,'objective_achieved':False,'rows':len(observed),'field_count':len(observed.columns),'maximum_reconstruction_error':float(np.nanmax(np.abs(restored-original))),'unmodified_other_fields':len(other),'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','no_new_predictive_information':True,'frozen_model_inputs_changed':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/features','analysis/reconstructed','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='no_fit_lossless_momentum_coordinates',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,learned_parameters=0)
 rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit,aliases=aliases)
 rec.save_objects(artifact_path='analysis',features=observed,reconstructed=restored,mapping=mapping,correlations=pd.DataFrame(rows),anchor=anchor,summary=summary)
 rec.save_objects(artifact_path='sources',feature_contract=(root/'src/data/allocation_feature_contract.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),maximum_reconstruction_error=summary['maximum_reconstruction_error'])
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');mapping.to_csv(card/'feature_mapping.csv',index=False);anchor.to_csv(card/'anchor_segments.csv',index=False);pd.DataFrame(rows).to_csv(card/'redundancy_diagnostic.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
print(pd.DataFrame(rows).to_string(index=False));print(anchor.to_string(index=False))


# 动量重叠拆分：信息不增加，语义更清楚

本轮0拟合、0回测、0未来收益相关检验、0参数搜索。延续同义字段去重后的52字段研究视图，仅改变其中SOXL/SOXX各5个累计动量的坐标表达，其他42字段逐值不变。

固定沿用原有3/5/10/20/60窗口端点：L(h)=log(1+r(h))，分段为L(3)、L(5)-L(3)、L(10)-L(5)、L(20)-L(10)、L(60)-L(20)。它们分别对应最近1–3、4–5、6–10、11–20、21–60个收益区间；不新增周期、不估计权重。累计相加再expm1可以还原原值。缺失和非法收益保留/报错，不偷偷补值。

3966行经原生Qlib Dataset读取并检查，最大还原误差1.78e-15。研究字段数仍为52，冻结网络原输入、参数、生产release未改变。改变坐标不等于旧模型能直接使用新字段；禁止替换旧模型同名输入。

同一资产五个动量字段的平均绝对相关性：训练历史SOXX 0.49685→0.04265，SOXL 0.50369→0.03056；2023+复用历史SOXX 0.48474→0.04014，SOXL 0.48570→0.03496。样本数分别3020/946。只比较输入相关性，没有标签筛选；非重叠不等于统计独立，低相关不证明预测更好或模型过拟合已降低，跨资产/其他特征冗余仍存在。

2023-11-03：SOXX最近3日+7.4301%，第4–5日-0.4366%，第6–10日-2.8341%，第11–20日-4.7690%，第21–60日-3.7271%。因此该日同时具有短期强修复和此前不同区间持续下跌，不能把它合并为单一“趋势已恢复”标签。原模型75%现金是否合理还需独立决策证据，本分解不能反证风险控制或产生增仓命令。

决定：lossless_research_representation_only_no_alpha_claim。完成特征表达改进，未获得新的预测信息、净超额或独立前向证据。下一步核查风险/回撤特征是否也把不同经济含义混为一谈；只做语义与来源核查，不为提高相关性或回测结果选坐标/周期。

2项特征合同检查通过，原生Dataset读回、可逆性、42项未变字段与NAS证据回读通过。Recorder `b850876d31864010ae0d53d817d3231a`。

[固定映射](feature_mapping.csv) · [锚点历史分段](anchor_segments.csv) · [全区间冗余检查](redundancy_diagnostic.csv)
